In [ ]:
# Hücre 1: Kurulum
!pip install -q sentence-transformers faiss-cpu transformers accelerate
print(" Kütüphaneler kuruldu.")

 Kütüphaneler kuruldu.


In [ ]:
# Hücre 2 (REVİZE): Çok dilli embedding modeli
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Bilgi tabanı (her tekniğe hem Türkçe hem İngilizce açıklama ekliyoruz)
mitre_knowledge = [
    {
        "id": "T1110",
        "tr": "T1110 Brute Force: Aynı IP adresinden çok sayıda başarısız giriş denemesi. Saldırgan sistematik olarak parola dener.",
        "en": "T1110 Brute Force: Adversaries may use brute force techniques to gain access to accounts when passwords are unknown. This involves trying many passwords."
    },
    {
        "id": "T1110.001",
        "tr": "T1110.001 Password Guessing: Saldırgan yaygın parolaları tek tek deneyerek hesaba erişmeye çalışır. Başarısız deneme sayısı orta seviyededir.",
        "en": "T1110.001 Password Guessing: Adversaries may guess passwords to attempt access to accounts using a repetitive mechanism."
    },
    {
        "id": "T1110.003",
        "tr": "T1110.003 Password Spraying: Tek bir parolayı çok sayıda kullanıcıya deneme saldırısı.",
        "en": "T1110.003 Password Spraying: Adversaries may use a single password against many accounts."
    },
    {
        "id": "T1078",
        "tr": "T1078 Valid Accounts: Geçerli bir hesabın ele geçirilmesi. Olağan dışı saatlerde başarılı giriş.",
        "en": "T1078 Valid Accounts: Adversaries may obtain and abuse credentials of existing accounts."
    },
    {
        "id": "TBD",
        "tr": "TBD Bilinmeyen Anomali: Model tarafından şüpheli bulundu ancak bilinen bir MITRE tekniğiyle eşleşmedi.",
        "en": "TBD Unknown Anomaly: Flagged by the anomaly model but not matching any known MITRE technique."
    },
]

# Çok dilli model yükle
print(" Çok dilli embedding modeli yükleniyor...")
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Hem TR hem EN metinlerini vektörleştir (daha zengin temsil)
kb_texts = []
for item in mitre_knowledge:
    kb_texts.append(item["tr"])
    kb_texts.append(item["en"])

kb_embeddings = embedder.encode(kb_texts, normalize_embeddings=True)

# FAISS index
index = faiss.IndexFlatIP(kb_embeddings.shape[1])
index.add(kb_embeddings)

print(f" Bilgi tabanı hazır. {len(mitre_knowledge)} teknik, {len(kb_texts)} metin vektörleştirildi.")

 Çok dilli embedding modeli yükleniyor...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

 Bilgi tabanı hazır. 5 teknik, 10 metin vektörleştirildi.


In [ ]:
# Hücre 3 (REVİZE): Retrieval fonksiyonu
def retrieve_mitre_context(query, k=1):
    """En ilgili MITRE tekniğini döndürür. Hem ID hem açıklama içerir."""
    query_emb = embedder.encode([query], normalize_embeddings=True)
    distances, indices = index.search(query_emb, k)

    # her teknik 2 vektör (TR + EN) olduğu için index // 2
    seen_ids = []
    results = []
    for i in indices[0]:
        tech = mitre_knowledge[i // 2]
        if tech["id"] not in seen_ids:
            seen_ids.append(tech["id"])
            results.append(tech)

    return results

# Test
test_query = "Aynı IP adresinden çok sayıda başarısız giriş denemesi yapıldı."
result = retrieve_mitre_context(test_query)[0]
print(f" Sorgu: {test_query}\n")
print(f" En ilgili MITRE Tekniği: {result['id']}")
print(f" Açıklama: {result['tr']}")

 Sorgu: Aynı IP adresinden çok sayıda başarısız giriş denemesi yapıldı.

 En ilgili MITRE Tekniği: T1110
 Açıklama: T1110 Brute Force: Aynı IP adresinden çok sayıda başarısız giriş denemesi. Saldırgan sistematik olarak parola dener.


In [ ]:
# Hücre 3.5: Google Drive Bağlantısı ve Veriyi Yükleme
import pandas as pd
from google.colab import drive
import os

# Google Drive'ı bağla
if not os.path.exists("/content/drive"):
    print(" Google Drive bağlanıyor...")
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/SentinelAI"
file_path = f"{BASE}/data/processed/logs_classified.parquet"

if os.path.exists(file_path):
    df = pd.read_parquet(file_path)
    print(f" Veri yüklendi: {len(df)} satır")
    print(f" Brute Force sayısı: {(df['threat_type']=='Brute Force').sum()}")
else:
    print(f" Hata: Belirtilen yolda dosya bulunamadı: {file_path}")
    print("Lütfen dosya yolunu ve Drive klasör yapınızı kontrol edin.")

 Veri yüklendi: 2100 satır
 Brute Force sayısı: 100


In [ ]:
# Hücre 4 (FINAL v4): Kurumsal Kural Tabanlı Raporlama (LLM'siz, Halüsinasyonsuz)
def generate_analyst_report(alarm_row):
    # 1. RAG ile MITRE bağlamını çek
    tech = retrieve_mitre_context(f"Olay tipi: {alarm_row['threat_type']}")[0]
    mitre_id = tech['id']
    mitre_name = tech['tr'].split(':')[0] if ':' in tech['tr'] else tech['tr']

    # 2. Aksiyonları belirle (Kural tabanlı)
    if mitre_id == "T1110":
        aksiyonlar = (
            f"1. Kaynak IP adresini ({alarm_row['ip']}) güvenlik duvarında geçici olarak bloklayın.\n"
            f"2. '{alarm_row['user']}' hesabının şifresini derhal değiştirin ve Çok Faktörlü Kimlik Doğrulamayı (MFA) kontrol edin.\n"
            f"3. Aynı IP'den gelen diğer oturumları inceleyin."
        )
        analiz = (
            f"Sistem, {alarm_row['ip']} IP adresinden '{alarm_row['user']}' kullanıcısına yönelik "
            f"yoğun bir başarısız giriş denemesi ({alarm_row['attempts']} kez) tespit etti. "
            f"Bu durum, otomatik araçlarla yapılan ve hesabı ele geçirmeyi amaçlayan bir Brute Force (Kaba Kuvvet) saldırısıdır."
        )
    elif mitre_id == "T1078":
        aksiyonlar = (
            f"1. '{alarm_row['user']}' kullanıcısının gece yaptığı bu girişin kaynağını (VPN, uzak masaüstü vb.) doğrulayın.\n"
            f"2. Kullanıcının cihazında zararlı yazılım (malware) taraması başlatın.\n"
            f"3. Şüpheli bir durum varsa oturumu sonlandırın."
        )
        analiz = (
            f"'{alarm_row['user']}' kullanıcısı, olağan dışı bir saatte (gece) başarılı bir giriş yaptı. "
            f"Bu durum, saldırganın çalınmış geçerli kimlik bilgilerini (Valid Accounts) kullanarak sisteme sızdığını gösterebilir."
        )
    else:
        aksiyonlar = (
            "1. İlgili logları manuel olarak inceleyin.\n"
            "2. Kullanıcı davranış analitiği (UBA) verileriyle karşılaştırın."
        )
        analiz = (
            f"Modelimiz {alarm_row['ip']} IP adresinden gelen aktiviteyi anormal olarak işaretledi. "
            f"Ancak bu aktivite bilinen bir MITRE tekniğiyle eşleşmedi. Uzman incelemesi gereklidir."
        )

    # 3. Nihai Rapor (Markdown)
    rapor = f"""###  SentinelAI Olay Raporu

** Olay Özeti:**
{alarm_row['timestamp']} tarihinde sistem, **{alarm_row['ip']}** IP adresinden **'{alarm_row['user']}'** kullanıcısına yönelik bir anomali tespit etti.

** Yapay Zeka Analizi:**
{analiz}

** Eşleşen MITRE ATT&CK Tekniği:**
* **ID:** {mitre_id}
* **Teknik:** {mitre_name}

** Önerilen Aksiyonlar (SOC Playbook):**
{aksiyonlar}
"""
    return rapor, mitre_id


# --- TEST ---
# Brute Force alarmı
alarm_bf = df[df['threat_type'] == 'Brute Force'].iloc[0]
rapor_bf, _ = generate_analyst_report(alarm_bf)
print(rapor_bf)
print("\n" + "="*60 + "\n")

# Suspicious Login (T1078) alarmı için güvenli arama
cond1 = df['threat_type'] == 'Suspicious Login Time'
cond2 = df['mitre_id'] == 'T1078'

filtered_df = df[cond1] if cond1.any() else df[cond2]

if not filtered_df.empty:
    alarm_t1078 = filtered_df.iloc[0]
    rapor_t1078, _ = generate_analyst_report(alarm_t1078)
    print(rapor_t1078)
else:
    print("ℹ Veri kümesinde T1078 veya Suspicious Login Time tipinde bir alarma rastlanamadı.")

###  SentinelAI Olay Raporu

** Olay Özeti:**
2026-11-17 02:00:06 tarihinde sistem, **192.168.1.45** IP adresinden **'admin'** kullanıcısına yönelik bir anomali tespit etti.

** Yapay Zeka Analizi:**
Sistem, 192.168.1.45 IP adresinden 'admin' kullanıcısına yönelik yoğun bir başarısız giriş denemesi (28 kez) tespit etti. Bu durum, otomatik araçlarla yapılan ve hesabı ele geçirmeyi amaçlayan bir Brute Force (Kaba Kuvvet) saldırısıdır.

** Eşleşen MITRE ATT&CK Tekniği:**
* **ID:** T1110
* **Teknik:** T1110 Brute Force

** Önerilen Aksiyonlar (SOC Playbook):**
1. Kaynak IP adresini (192.168.1.45) güvenlik duvarında geçici olarak bloklayın.
2. 'admin' hesabının şifresini derhal değiştirin ve Çok Faktörlü Kimlik Doğrulamayı (MFA) kontrol edin.
3. Aynı IP'den gelen diğer oturumları inceleyin.



ℹ Veri kümesinde T1078 veya Suspicious Login Time tipinde bir alarma rastlanamadı.
